In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

# install necessary requirements into the Colab environment

In [2]:
# connect to Google Drive
from google.colab import drive

drive.mount('/content/drive')
drive_folder = '/content/drive/MyDrive/U5550685'

# path to saved lora adapters for the fine-tuned model
lora_path = os.path.join(drive_folder, "gpt_oss_qlora")

Mounted at /content/drive


In [3]:
# import necessary remaining libraries
from unsloth import FastLanguageModel

from peft import PeftModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
# the exploit type entered by the user to the wrapper script
# comment/uncomment as necessary

exploit_type = 'xss'
# exploit_type = 'sqli'

In [5]:
# initialise base model
max_seq_length = 2048 # long enough to read the sys prompt and user input
dtype = None

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gpt-oss-20b",
    dtype=dtype,
    max_seq_length=max_seq_length,
    load_in_4bit=True, # QLoRA
)

==((====))==  Unsloth 2026.5.2: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.37G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [6]:
# attach fine-tune weights from Google Drive to base model
model = PeftModel.from_pretrained(model, lora_path)
model.eval()
FastLanguageModel.for_inference(model)

Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GptOssForCausalLM(
      (model): GptOssModel(
        (embed_tokens): Embedding(201088, 2880, padding_idx=199999)
        (layers): ModuleList(
          (0-23): 24 x GptOssDecoderLayer(
            (self_attn): GptOssAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2880, out_features=4096, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2880, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lo

In [7]:
# define system prompts
XSS_SYSTEM_PROMPT = """You are an assistant to a penetration tester who is performing a retest for previously identified XSS vulnerability. The client has protected their website by implementing a WAF or using input validation and sanitisation.

You will be provided with a HTTP request that previously triggered XSS. This original request no longer works due to the defensive measures in place. Your task is to perform a mutation on the payload from the original request so that it will bypass the defensive measures and trigger the underlying XSS vulnerability.

You must:
    - Significantly mutate the original payload but ensure that it is still valid code that will execute
    - Output only the modified, valid HTTP request and nothing else
    - Modify only the payload in the request
    - Preserve the HTTP method, path, headers, and overall structure other than of the payload itself
    - Only use payload's that will trigger automatically
    - Only use payloads that set the log to the console a string of the format 'XSS_test_<number>' - where the number is the number from the original payload incremented by 1 (or 1 if no number was used in the original request)
    - Do not attempt to exploit other vulnerabilities, focus on XSS with a WAF/input validation/input sanitisation bypass

When modifying the payload to bypass the defensive measures, you must use one or more of the following:
    - Different tags and event handlers
    - Encoding e.g. Base64, URL, HTML entity - but ensure the payload created is still valid and will execute
    - Randomise capitalisation
    - Remove whitespace or break up predictable text patterns
    - Prepend/insert/append additional syntax to escape HTML contexts and avoid predictable patterns

You must significantly modify the payload but it should still execute!"""

SQLI_SYSTEM_PROMPT = """You are an assistant to a penetration tester who is performing a retest for previously identified SQLi vulnerability. The client has protected their website by implementing a WAF or using input validation and sanitisation.

You will be provided with a HTTP request that previously triggered SQLi. This original request no longer works due to the defensive measures in place. Your task is to perform a mutation on the payload from the original request so that it will bypass the defensive measures and trigger the underlying SQLi vulnerability.

You must:

    - Significantly mutate the original payload but ensure that it is still valid code that will execute
    - Output only the modified, valid HTTP request and nothing else
    - Modify only the payload in the request
    - Preserve the HTTP method, path, headers, and overall structure other than of the payload itself
    - Only use payloads that either cause the site to take 2 seconds to load (e.g. Sleep(2)) or display a string of the format 'SQLi_test_<number>' - where the number is the number from the original payload incremented by 1 (or 1 if no number was used in the original request)
    - Do not attempt to exploit other vulnerabilities, focus on SQLi with a WAF/input validation/input sanitisation bypass
    - Maintain compatibility with the original SQL context (e.g. DB type, column count), but you must significantly alter the payload so it will bypass the defensive measures in place

When modifying the payload to bypass the defensive measures, you must use one or more of the following:
    - Encoding e.g. using character codes or XML encoding - but ensure the payload created is still valid and will execute
    - Random capitalisation
    - Backslash escape quotations
    - Change the comment character used to another compatible character
    - Change whitespace to other characters
    - Finish the query with a '
    - Use comments to break up predictable text patterns
    - Use executable comments

You must significantly modify the payload but it should still execute!"""

In [8]:
# fetch the input request from Google Drive
REDACTED_FILE = os.path.join(drive_folder, "redacted_request.txt")
with open(REDACTED_FILE, "r") as f:
    previous_request = f.read()

In [9]:
# function to format messages as required
def to_messages(previous_request, exploit_type):
    if exploit_type == "xss":
        messages = [
            {"role": "system", "content": XSS_SYSTEM_PROMPT},
            {"role": "user", "content": previous_request}
        ]
    elif exploit_type == "sqli":
        messages = [
            {"role": "system", "content": SQLI_SYSTEM_PROMPT},
            {"role": "user", "content": previous_request}
        ]
    return messages

In [10]:
# convert the input to tokens
messages = to_messages(previous_request, exploit_type)

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low",
).to("cuda")

In [11]:
# generate an output
outputs = model.generate(
    **inputs,
    max_new_tokens=1000,
    do_sample=True, # non-deterministic
    temperature=0.7,
    top_p=0.6,
    top_k=10
)

# only return the newly generated tokens - i.e. not the input
generated = outputs[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(generated, skip_special_tokens=True)
# remove the model's thought process from the output
response = response.split("assistantfinal", 1)[-1].strip()

In [12]:
# save the generated output to a file on Google Drive
GENERATED_FILE = os.path.join(drive_folder, "generated_request.txt")
with open(GENERATED_FILE, "w") as f:
    f.write(response)